# E3. Letter Carry-Over Analysis: Trial n-1 → Trial n

Do trials on which **the current probe letter** (trial *n* probe target) appeared in the **trial *n−1* encoded memory set** differ from trials where it did not?

For each participant we contrast **probe in n-1** vs **probe not in n-1** trials on four outcomes:

1. Probe (memory-recognition) accuracy on trial *n*
2. Go RT on trial *n*
3. Number of stop trials per condition (descriptive)
4. SSRT (mean Go RT − mean SSD) per condition

## Table of Contents
1. Setup & Data Loading
2. Within-Subject CI Function
3. Compute Probe-in-(n-1) Labels
4. Probe Accuracy: In n-1 vs not
5. Go RT: In n-1 vs not
6. Stop-Trial Counts: In n-1 vs not
7. SSRT: In n-1 vs not
8. By Memory Load — all outcomes above are also stratified by trial *n* encoding load (`memory_trial_stimLength`)


In [ ]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import scipy.stats as stats

from stop_wm.config import ProjectConfig

# Initialize config — go up one level from notebooks/ to project root
project_root = Path.cwd().parent
config = ProjectConfig(project_root=project_root)

# Pathing
trial_wise_data_path = config.results_dir / 'post_qc_stop_signal_wm_trials.csv'

# Load data
trial_wise_data = pd.read_csv(trial_wise_data_path)
print(f"Loaded {len(trial_wise_data)} trials from {trial_wise_data['participant_id'].nunique()} participants")


## 2. Within-Subject Confidence Interval Function

Cousineau (2005) + Morey (2008) within-subject CIs for repeated-measures comparisons.


In [ ]:
from stop_wm.within_subject_ci import calculate_within_subject_ci


## 3. Compute Probe-in-(n-1) Labels

For every trial *n* within a participant×block we take trial *n−1*'s encoded letters from `memory_trial_stim` and the **probe target letter** on trial *n* from `memory_recognition_recognitionLetter`.

**Inclusion rules:**
- Trial *n* must have at least one prior trial in the same block (no carry-over across blocks).
- Trial *n* must have a valid single-letter probe target; trial *n−1* must have a non-empty encoding set. Otherwise the label is undefined (excluded).

The column `letter_shared` is retained for downstream cells; here it means **probe letter ∈ trial n−1 encoding set** (not overlap between the full encoding sets on *n* and *n−1*).


In [ ]:
# === PROBE-IN-TRIAL-N-MINUS-ONE FLAGS ===
# `letter_shared`: True iff the probe target on trial n (`memory_recognition_recognitionLetter`)
# appears in the encoding set from trial n-1 (shifted `memory_trial_stim`).

df = trial_wise_data.copy()

# Sort so .shift(1) gives the immediately preceding trial within block
df = df.sort_values(['participant_id', 'block_num', 'current_trial']).reset_index(drop=True)

def _clean_set(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s in ('()', 'nan', ''):
        return ''
    return ''.join(ch for ch in s if ch.isalpha()).upper()

def _clean_probe_letter(x):
    """Single target letter on the probe phase of trial n; () / missing → empty."""
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s in ('()', 'nan', ''):
        return ''
    letters = ''.join(ch for ch in s if ch.isalpha()).upper()
    if len(letters) == 1:
        return letters
    return ''

df['curr_set'] = df['memory_trial_stim'].map(_clean_set)
df['prev_set'] = df.groupby(['participant_id', 'block_num'])['curr_set'].shift(1)
df['prev_set'] = df['prev_set'].fillna('')

df['probe_letter'] = df['memory_recognition_recognitionLetter'].map(_clean_probe_letter)

def _probe_in_prev(row):
    probe = row['probe_letter']
    prev = row['prev_set']
    if not probe or not prev:
        return np.nan
    return probe in set(prev)

df['letter_shared'] = df.apply(_probe_in_prev, axis=1)

# Keep only trials with a well-defined comparison
df_lc = df[df['letter_shared'].notna()].copy()
df_lc['letter_shared'] = df_lc['letter_shared'].astype(bool)

print("="*70)
print("Probe-in-(n-1) labeling")
print("="*70)
print(f"Total trials in raw data:                      {len(df):>8d}")
print(f"Trials with valid probe vs n-1 set:           {len(df_lc):>8d}")
print(f"  → probe letter ∈ trial n-1 encoding set:    {int(df_lc['letter_shared'].sum()):>8d}")
print(f"  → probe letter ∉ trial n-1 encoding set:    {int((~df_lc['letter_shared']).sum()):>8d}")
print(f"Participants represented:                      {df_lc['participant_id'].nunique():>8d}")

# WM load on trial n (encoding set size); used for stratified analyses below
LOAD_COL = 'memory_trial_stimLength'


## 4. Probe Accuracy: In n-1 vs not

Probe accuracy is mean `memory_recognition_correct_trial`. Following the project convention (see CLAUDE.md, *Commission vs. omission errors*), omissions (`memory_recognition_rt` NaN) are **excluded** so the metric reflects commission accuracy only.

Statistic: paired *t*-test on participant-level means (probe in n-1 vs not) with Cohen's *d*. Bar plot uses within-subject 95 % CIs.


In [ ]:
# === PROBE ACCURACY: PROBE IN N-1 vs NOT ===

# Commission only: drop omissions
probe_df = df_lc.loc[df_lc['memory_recognition_rt'].notna()].copy()

probe_acc_subj = (
    probe_df
    .groupby(['participant_id', 'letter_shared'])['memory_recognition_correct_trial']
    .mean()
    .unstack('letter_shared')
    .rename(columns={False: 'Not in n-1', True: 'In n-1'})
    .dropna(subset=['Not in n-1', 'In n-1'])
)

print(f"Participants with complete data for both conditions: {len(probe_acc_subj)}")

# Means + within-subject CIs
probe_acc_means = probe_acc_subj[['Not in n-1', 'In n-1']].mean().values
probe_acc_cis = calculate_within_subject_ci(probe_acc_subj[['Not in n-1', 'In n-1']])

# Paired test
t_stat, p_val = stats.ttest_rel(probe_acc_subj['In n-1'], probe_acc_subj['Not in n-1'])
diff = probe_acc_subj['In n-1'] - probe_acc_subj['Not in n-1']
cohen_d = diff.mean() / diff.std(ddof=1)

print("="*70)
print("Paired t-test: Probe Accuracy (In n-1 vs Not in n-1)")
print("="*70)
print(f"t({len(probe_acc_subj) - 1}) = {t_stat:.4f}, p = {p_val:.6f}")
print(f"Significant: {'Yes' if p_val < 0.05 else 'No'} (α = 0.05)")
print(f"Effect size (Cohen's d): {cohen_d:.4f}")

# === PLOT ===
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x_pos = np.arange(2)
colors = ['#2E86AB', '#A23B72']
bars = ax.bar(x_pos, probe_acc_means, yerr=probe_acc_cis, capsize=8,
              alpha=0.8, color=colors, edgecolor='black', linewidth=1.5,
              error_kw={'linewidth': 2, 'ecolor': 'black'})
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')

ax.set_xticks(x_pos)
ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=11)
ax.set_ylabel('Probe Accuracy', fontsize=13, fontweight='bold')
ax.set_xlabel('Current probe letter in trial n−1 memory set', fontsize=13, fontweight='bold')
ax.set_title('E3. Probe accuracy: probe in trial n−1 set vs not\n(Within-Subject 95% CI)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

for bar, mean, ci in zip(bars, probe_acc_means, probe_acc_cis):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# ---- By memory load (encoding load on trial n) ----
_loads = sorted(probe_df[LOAD_COL].dropna().unique())
fig, axes = plt.subplots(1, len(_loads), figsize=(4 * len(_loads), 5), sharey=True, squeeze=False)
axes = axes.ravel()
for ax, load in zip(axes, _loads):
    sub = probe_df[probe_df[LOAD_COL] == load]
    probe_acc_subj_ld = (
        sub
        .groupby(['participant_id', 'letter_shared'])['memory_recognition_correct_trial']
        .mean()
        .unstack('letter_shared')
        .rename(columns={False: 'Not in n-1', True: 'In n-1'})
        .dropna(subset=['Not in n-1', 'In n-1'])
    )
    n_ld = len(probe_acc_subj_ld)
    if n_ld < 2:
        ax.text(0.5, 0.5, f'Load {load}\nInsufficient paired data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Load {int(load)} (n={n_ld})')
        continue
    means_ld = probe_acc_subj_ld[['Not in n-1', 'In n-1']].mean().values
    cis_ld = calculate_within_subject_ci(probe_acc_subj_ld[['Not in n-1', 'In n-1']])
    t_ld, p_ld = stats.ttest_rel(probe_acc_subj_ld['In n-1'], probe_acc_subj_ld['Not in n-1'])
    print(f"\n--- Probe accuracy, load {load} (n={n_ld}) ---")
    print(f"  t({n_ld - 1}) = {t_ld:.4f}, p = {p_ld:.6f}")
    x_pos = np.arange(2)
    bars = ax.bar(x_pos, means_ld, yerr=cis_ld, capsize=6, alpha=0.8,
                  color=['#2E86AB', '#A23B72'], edgecolor='black', linewidth=1.2,
                  error_kw={'linewidth': 1.5, 'ecolor': 'black'})
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=9)
    ax.set_ylabel('Probe accuracy', fontsize=10)
    ax.set_title(f'Load {int(load)} (n={n_ld})\np={p_ld:.4f}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, mean, ci in zip(bars, means_ld, cis_ld):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('E3. Probe accuracy by memory load\n(within-subject 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Go RT: In n-1 vs not

Mean `stop_trial_rt` on go trials only (`stop_trial_condition == 'go'`). Omissions (no key pressed) are automatically excluded because their RT is NaN.


In [ ]:
# === GO RT: PROBE IN N-1 vs NOT ===

go_df = df_lc[df_lc['stop_trial_condition'] == 'go'].copy()

go_rt_subj = (
    go_df
    .dropna(subset=['stop_trial_rt'])
    .groupby(['participant_id', 'letter_shared'])['stop_trial_rt']
    .mean()
    .unstack('letter_shared')
    .rename(columns={False: 'Not in n-1', True: 'In n-1'})
    .dropna(subset=['Not in n-1', 'In n-1'])
)

print(f"Participants with complete data for both conditions: {len(go_rt_subj)}")

go_rt_means = go_rt_subj[['Not in n-1', 'In n-1']].mean().values
go_rt_cis = calculate_within_subject_ci(go_rt_subj[['Not in n-1', 'In n-1']])

t_stat, p_val = stats.ttest_rel(go_rt_subj['In n-1'], go_rt_subj['Not in n-1'])
diff = go_rt_subj['In n-1'] - go_rt_subj['Not in n-1']
cohen_d = diff.mean() / diff.std(ddof=1)

print("="*70)
print("Paired t-test: Go RT (In n-1 vs Not in n-1)")
print("="*70)
print(f"t({len(go_rt_subj) - 1}) = {t_stat:.4f}, p = {p_val:.6f}")
print(f"Significant: {'Yes' if p_val < 0.05 else 'No'} (α = 0.05)")
print(f"Effect size (Cohen's d): {cohen_d:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x_pos = np.arange(2)
colors = ['#2E86AB', '#A23B72']
bars = ax.bar(x_pos, go_rt_means, yerr=go_rt_cis, capsize=8,
              alpha=0.8, color=colors, edgecolor='black', linewidth=1.5,
              error_kw={'linewidth': 2, 'ecolor': 'black'})
ax.set_xticks(x_pos)
ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=11)
ax.set_ylabel('Go RT (ms)', fontsize=13, fontweight='bold')
ax.set_xlabel('Current probe letter in trial n−1 memory set', fontsize=13, fontweight='bold')
ax.set_title('E3. Go RT: probe in trial n−1 set vs not\n(Within-Subject 95% CI)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, mean, ci in zip(bars, go_rt_means, go_rt_cis):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 5,
            f'{mean:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# ---- By memory load ----
_loads = sorted(go_df[LOAD_COL].dropna().unique())
fig, axes = plt.subplots(1, len(_loads), figsize=(4 * len(_loads), 5), sharey=False, squeeze=False)
axes = axes.ravel()
for ax, load in zip(axes, _loads):
    sub = go_df[(go_df[LOAD_COL] == load) & go_df['stop_trial_rt'].notna()]
    go_rt_subj_ld = (
        sub
        .groupby(['participant_id', 'letter_shared'])['stop_trial_rt']
        .mean()
        .unstack('letter_shared')
        .rename(columns={False: 'Not in n-1', True: 'In n-1'})
        .dropna(subset=['Not in n-1', 'In n-1'])
    )
    n_ld = len(go_rt_subj_ld)
    if n_ld < 2:
        ax.text(0.5, 0.5, f'Load {load}\nInsufficient paired data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Load {int(load)} (n={n_ld})')
        continue
    means_ld = go_rt_subj_ld[['Not in n-1', 'In n-1']].mean().values
    cis_ld = calculate_within_subject_ci(go_rt_subj_ld[['Not in n-1', 'In n-1']])
    t_ld, p_ld = stats.ttest_rel(go_rt_subj_ld['In n-1'], go_rt_subj_ld['Not in n-1'])
    print(f"\n--- Go RT, load {load} (n={n_ld}) ---")
    print(f"  t({n_ld - 1}) = {t_ld:.4f}, p = {p_ld:.6f}")
    x_pos = np.arange(2)
    bars = ax.bar(x_pos, means_ld, yerr=cis_ld, capsize=6, alpha=0.8,
                  color=['#2E86AB', '#A23B72'], edgecolor='black', linewidth=1.2,
                  error_kw={'linewidth': 1.5, 'ecolor': 'black'})
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=9)
    ax.set_ylabel('Go RT (ms)', fontsize=10)
    ax.set_title(f'Load {int(load)} (n={n_ld})\np={p_ld:.4f}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, mean, ci in zip(bars, means_ld, cis_ld):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 5,
                f'{mean:.1f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('E3. Go RT by memory load\n(within-subject 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Stop-Trial Counts: In n-1 vs not

Descriptive: how many stop trials does each participant have in each carry-over condition? This is informative for interpreting SSRT (next section), since very few stop trials in a cell makes its SSRT noisy.


In [ ]:
# === STOP TRIAL COUNTS ===

stop_df = df_lc[df_lc['stop_trial_condition'] == 'stop'].copy()

stop_counts_subj = (
    stop_df
    .groupby(['participant_id', 'letter_shared'])
    .size()
    .unstack('letter_shared', fill_value=0)
    .rename(columns={False: 'Not in n-1', True: 'In n-1'})
)
# Make sure both columns exist
for col in ['Not in n-1', 'In n-1']:
    if col not in stop_counts_subj.columns:
        stop_counts_subj[col] = 0
stop_counts_subj = stop_counts_subj[['Not in n-1', 'In n-1']]

print(f"Participants represented: {len(stop_counts_subj)}")
print("\nStop-trial counts per condition:")
print(stop_counts_subj.describe().round(2))

stop_count_means = stop_counts_subj.mean().values
stop_count_cis = calculate_within_subject_ci(stop_counts_subj)

t_stat, p_val = stats.ttest_rel(stop_counts_subj['In n-1'], stop_counts_subj['Not in n-1'])
diff = stop_counts_subj['In n-1'] - stop_counts_subj['Not in n-1']
cohen_d = diff.mean() / diff.std(ddof=1)

print("="*70)
print("Paired t-test: Stop-Trial Count (In n-1 vs Not in n-1)")
print("="*70)
print(f"t({len(stop_counts_subj) - 1}) = {t_stat:.4f}, p = {p_val:.6f}")
print(f"Effect size (Cohen's d): {cohen_d:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x_pos = np.arange(2)
colors = ['#2E86AB', '#A23B72']
bars = ax.bar(x_pos, stop_count_means, yerr=stop_count_cis, capsize=8,
              alpha=0.8, color=colors, edgecolor='black', linewidth=1.5,
              error_kw={'linewidth': 2, 'ecolor': 'black'})
ax.set_xticks(x_pos)
ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=11)
ax.set_ylabel('Stop Trials per Participant', fontsize=13, fontweight='bold')
ax.set_xlabel('Current probe letter in trial n−1 memory set', fontsize=13, fontweight='bold')
ax.set_title('E3. Stop-trial count: probe in trial n−1 set vs not\n(Within-Subject 95% CI)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, mean, ci in zip(bars, stop_count_means, stop_count_cis):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 0.5,
            f'{mean:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# ---- By memory load ----
_loads = sorted(stop_df[LOAD_COL].dropna().unique())
fig, axes = plt.subplots(1, len(_loads), figsize=(4 * len(_loads), 5), sharey=False, squeeze=False)
axes = axes.ravel()
for ax, load in zip(axes, _loads):
    sub = stop_df[stop_df[LOAD_COL] == load]
    stop_counts_subj_ld = (
        sub
        .groupby(['participant_id', 'letter_shared'])
        .size()
        .unstack('letter_shared', fill_value=0)
        .rename(columns={False: 'Not in n-1', True: 'In n-1'})
    )
    for col in ['Not in n-1', 'In n-1']:
        if col not in stop_counts_subj_ld.columns:
            stop_counts_subj_ld[col] = 0
    stop_counts_subj_ld = stop_counts_subj_ld[['Not in n-1', 'In n-1']]
    n_ld = len(stop_counts_subj_ld)
    if n_ld < 2:
        ax.text(0.5, 0.5, f'Load {load}\nInsufficient paired data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Load {int(load)} (n={n_ld})')
        continue
    means_ld = stop_counts_subj_ld.mean().values
    cis_ld = calculate_within_subject_ci(stop_counts_subj_ld)
    t_ld, p_ld = stats.ttest_rel(stop_counts_subj_ld['In n-1'], stop_counts_subj_ld['Not in n-1'])
    print(f"\n--- Stop-trial count, load {load} (n={n_ld}) ---")
    print(f"  t({n_ld - 1}) = {t_ld:.4f}, p = {p_ld:.6f}")
    x_pos = np.arange(2)
    bars = ax.bar(x_pos, means_ld, yerr=cis_ld, capsize=6, alpha=0.8,
                  color=['#2E86AB', '#A23B72'], edgecolor='black', linewidth=1.2,
                  error_kw={'linewidth': 1.5, 'ecolor': 'black'})
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=9)
    ax.set_ylabel('Stop trials (count)', fontsize=10)
    ax.set_title(f'Load {int(load)} (n={n_ld})\np={p_ld:.4f}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, mean, ci in zip(bars, means_ld, cis_ld):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 0.5,
                f'{mean:.1f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('E3. Stop-trial counts by memory load\n(within-subject 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. SSRT: In n-1 vs not

**Mean method:** SSRT = mean Go RT − mean SSD, computed per participant × condition.

- Mean Go RT uses all go trials with a recorded RT in the condition.
- Mean SSD uses all stop trials in the condition (regardless of stopping outcome).
- Participants are required to have ≥ 1 go-RT and ≥ 1 SSD in **both** conditions; otherwise dropped.

Note (see Section 6): cells with very few stop trials yield noisy SSRT estimates. Treat group-level SSRT differences cautiously if **In n-1** or **Not in n-1** stop-trial counts are small.


In [ ]:
# === SSRT (MEAN METHOD): PROBE IN N-1 vs NOT ===

# Per-condition mean Go RT
go_rt_per_cond = (
    df_lc[df_lc['stop_trial_condition'] == 'go']
    .dropna(subset=['stop_trial_rt'])
    .groupby(['participant_id', 'letter_shared'])['stop_trial_rt']
    .mean()
)

# Per-condition mean SSD
ssd_per_cond = (
    df_lc[df_lc['stop_trial_condition'] == 'stop']
    .dropna(subset=['stop_trial_SSD'])
    .groupby(['participant_id', 'letter_shared'])['stop_trial_SSD']
    .mean()
)

ssrt_long = pd.concat([
    go_rt_per_cond.rename('mean_go_rt'),
    ssd_per_cond.rename('mean_ssd'),
], axis=1)
ssrt_long['SSRT'] = ssrt_long['mean_go_rt'] - ssrt_long['mean_ssd']

ssrt_subj = (
    ssrt_long['SSRT']
    .unstack('letter_shared')
    .rename(columns={False: 'Not in n-1', True: 'In n-1'})
    .dropna(subset=['Not in n-1', 'In n-1'])
)

print(f"Participants with SSRT for both conditions: {len(ssrt_subj)}")
print("\nPer-condition SSRT summary:")
print(ssrt_subj.describe().round(2))

ssrt_means = ssrt_subj[['Not in n-1', 'In n-1']].mean().values
ssrt_cis = calculate_within_subject_ci(ssrt_subj[['Not in n-1', 'In n-1']])

t_stat, p_val = stats.ttest_rel(ssrt_subj['In n-1'], ssrt_subj['Not in n-1'])
diff = ssrt_subj['In n-1'] - ssrt_subj['Not in n-1']
cohen_d = diff.mean() / diff.std(ddof=1)

print("="*70)
print("Paired t-test: SSRT (In n-1 vs Not in n-1)")
print("="*70)
print(f"t({len(ssrt_subj) - 1}) = {t_stat:.4f}, p = {p_val:.6f}")
print(f"Significant: {'Yes' if p_val < 0.05 else 'No'} (α = 0.05)")
print(f"Effect size (Cohen's d): {cohen_d:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x_pos = np.arange(2)
colors = ['#2E86AB', '#A23B72']
bars = ax.bar(x_pos, ssrt_means, yerr=ssrt_cis, capsize=8,
              alpha=0.8, color=colors, edgecolor='black', linewidth=1.5,
              error_kw={'linewidth': 2, 'ecolor': 'black'})
ax.set_xticks(x_pos)
ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=11)
ax.set_ylabel('SSRT (ms)', fontsize=13, fontweight='bold')
ax.set_xlabel('Current probe letter in trial n−1 memory set', fontsize=13, fontweight='bold')
ax.set_title('E3. SSRT: probe in trial n−1 set vs not\n(Within-Subject 95% CI)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, mean, ci in zip(bars, ssrt_means, ssrt_cis):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 3,
            f'{mean:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# ---- By memory load ----
_loads = sorted(df_lc[LOAD_COL].dropna().unique())
fig, axes = plt.subplots(1, len(_loads), figsize=(4 * len(_loads), 5), sharey=False, squeeze=False)
axes = axes.ravel()
for ax, load in zip(axes, _loads):
    base = df_lc[df_lc[LOAD_COL] == load]
    go_rt_per_cond_ld = (
        base[base['stop_trial_condition'] == 'go']
        .dropna(subset=['stop_trial_rt'])
        .groupby(['participant_id', 'letter_shared'])['stop_trial_rt']
        .mean()
    )
    ssd_per_cond_ld = (
        base[base['stop_trial_condition'] == 'stop']
        .dropna(subset=['stop_trial_SSD'])
        .groupby(['participant_id', 'letter_shared'])['stop_trial_SSD']
        .mean()
    )
    ssrt_long_ld = pd.concat([
        go_rt_per_cond_ld.rename('mean_go_rt'),
        ssd_per_cond_ld.rename('mean_ssd'),
    ], axis=1)
    ssrt_long_ld['SSRT'] = ssrt_long_ld['mean_go_rt'] - ssrt_long_ld['mean_ssd']
    ssrt_subj_ld = (
        ssrt_long_ld['SSRT']
        .unstack('letter_shared')
        .rename(columns={False: 'Not in n-1', True: 'In n-1'})
        .dropna(subset=['Not in n-1', 'In n-1'])
    )
    n_ld = len(ssrt_subj_ld)
    if n_ld < 2:
        ax.text(0.5, 0.5, f'Load {load}\nInsufficient paired data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Load {int(load)} (n={n_ld})')
        continue
    means_ld = ssrt_subj_ld[['Not in n-1', 'In n-1']].mean().values
    cis_ld = calculate_within_subject_ci(ssrt_subj_ld[['Not in n-1', 'In n-1']])
    t_ld, p_ld = stats.ttest_rel(ssrt_subj_ld['In n-1'], ssrt_subj_ld['Not in n-1'])
    print(f"\n--- SSRT, load {load} (n={n_ld}) ---")
    print(f"  t({n_ld - 1}) = {t_ld:.4f}, p = {p_ld:.6f}")
    x_pos = np.arange(2)
    bars = ax.bar(x_pos, means_ld, yerr=cis_ld, capsize=6, alpha=0.8,
                  color=['#2E86AB', '#A23B72'], edgecolor='black', linewidth=1.2,
                  error_kw={'linewidth': 1.5, 'ecolor': 'black'})
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Not in n-1', 'In n-1'], fontsize=9)
    ax.set_ylabel('SSRT (ms)', fontsize=10)
    ax.set_title(f'Load {int(load)} (n={n_ld})\np={p_ld:.4f}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, mean, ci in zip(bars, means_ld, cis_ld):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + ci + 3,
                f'{mean:.1f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('E3. SSRT by memory load\n(within-subject 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
